You have recently joined MediAssist AI Pvt. Ltd. as a Generative AI Engineer. The company is developing an AI assistant for hospitals that helps doctors summarize patient reports, answer clinical questions, and generate discharge summaries.

The current chatbot has several problems:

• Gives inconsistent responses
• Hallucinates medical information
• Produces unstructured outputs
• Cannot switch between different LLMs
• Difficult to maintain because prompts are poorly written
Your manager asks you to redesign the solution using Hugging Face Transformers and Prompt Engineering best practices.

# Task 1 — Hugging Face Integration

Build a chatbot using Hugging Face.

### Requirements

- Load any instruction-tuned model from Hugging Face.
- Use `AutoTokenizer`.
- Use `AutoModelForCausalLM`.
- Generate responses using the Transformers `pipeline`.
- Allow the user to ask medical questions interactively.

### Example

**User:**

Patient has fever for 3 days.

**Assistant:**

Possible causes...

In [3]:
!pip install transformers


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

chatBot = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

system_prompt = (
    "You are a helpful healthcare assistant. "
    "Answer general medical questions in simple language. "
    "Do not diagnose diseases or prescribe medicines."
)

while True:
    user_input = input("User: ")

    if user_input.lower() == "exit":
        print("Assistant: Bye")
        break

    prompt = f"{system_prompt}\n\nUser: {user_input}\nAssistant:"

    response = chatBot(
        prompt,
        max_new_tokens=150,
        do_sample=False
    )

    answer = response[0]["generated_text"][len(prompt):]

    print("Assistant:", answer.strip())

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

User: I have fever from 3 days


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: A fever lasting for three days can be concerning. It's important to stay hydrated and rest. You might want to take over-the-counter medications like acetaminophen or ibuprofen to help reduce your fever and any discomfort. However, if your fever persists, gets very high, or is accompanied by other symptoms like severe headache, stiff neck, rash, difficulty breathing, or persistent vomiting, it's important to seek medical attention right away. These could be signs of something more serious. Keep an eye on how you feel and monitor for any changes that might need medical care.
User: I am feeling cold too 


[transformers] Both `max_new_tokens` (=150) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Assistant: Feeling cold can be due to several reasons, such as the weather being cooler than usual, being in a room that's not well-heated, or even just feeling generally unwell. If you're feeling cold and it's not related to the temperature outside, you might want to check if you have a fever or if you're feeling under the weather. It's also important to make sure you're dressed warmly enough for the current conditions. If you're concerned about your health, it might be a good idea to consult with a healthcare provider.
User: exit
Assistant: Bye


# Task 2 — Prompt Engineering

The following prompt is performing poorly.

**Tell me about diabetes.**

Rewrite it using:

- Role Prompting
- Context
- Constraints
- Output Formatting

The response should contain:

- Disease Overview
- Symptoms
- Risk Factors
- Prevention
- When to Consult a Doctor

**Output must be in Markdown.**

In [8]:
prompt = """
You are MediAssist AI, a professional healthcare assistant.

Context:
The user wants to learn about a medical condition for educational purposes only.

Constraints:
- Provide only general medical information.
- Do not diagnose diseases.
- Do not prescribe medicines.
- Use simple and easy-to-understand language.
- Keep the response concise and accurate.

Output Format (Markdown):

# Disease Overview

# Symptoms

# Risk Factors

# Prevention

# When to Consult a Doctor

# Disclaimer

Question:
"""

# Take input from the user
user_question = input("Enter your medical related question: ")

# Append the user's question to the prompt
final_prompt = prompt + user_question

# Generate response
response = chatBot(
    final_prompt,
    max_new_tokens=300,
    do_sample=False
)

# Print response
generated_text = response[0]["generated_text"]
answer = generated_text[len(final_prompt):].strip()
print(answer)


Enter your medical related question: Tell me about diabetes


[transformers] Both `max_new_tokens` (=300) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


. # Disease Overview

Diabetes is a chronic condition that affects how your body uses blood sugar (glucose). It can lead to high levels of glucose in the blood, which can cause various health problems if not managed properly.

# Symptoms

Symptoms of diabetes can include frequent urination, increased thirst, extreme hunger, unusual weight loss, fatigue, irritability, and blurry vision. However, some people with diabetes may have no symptoms at all until their blood sugar levels become very high.

# Risk Factors

Risk factors for developing diabetes include being overweight or obese, having a family history of diabetes, being physically inactive, having high blood pressure, and having high cholesterol levels. Additionally, certain ethnicities, such as African Americans, Hispanic Americans, American Indians, and Asian Americans, are at higher risk.

# Prevention

While it's not always possible to prevent diabetes, you can reduce your risk by maintaining a healthy weight, eating a balance

# Task 3 — Few-Shot Prompting

Create a **Few-Shot prompt** that converts **clinical notes** into **structured summaries**.

### Example

**Input:**

```text
Patient has cough and fever.
```

**Output:**

```text
Symptoms:
- Cough
- Fever

Severity:
Moderate
```

### Requirements

- Provide **at least three examples** before the final input.
- The model should generate a structured summary based on the provided clinical notes.

In [11]:

few_shot_examples = """
Example 1

Input:
Patient has cough and fever.

Output:
Symptoms:
- Cough
- Fever

Severity:
Moderate

----------------------------------------

Example 2

Input:
Patient complains of headache, nausea, and dizziness.

Output:
Symptoms:
- Headache
- Nausea
- Dizziness

Severity:
Mild

----------------------------------------

Example 3

Input:
Patient has chest pain, shortness of breath, and oxygen level of 88%.

Output:
Symptoms:
- Chest pain
- Shortness of breath

Severity:
Severe

----------------------------------------
"""

patient_note = input("Enter Clinical Notes: ")

final_prompt = f"""
You are MediAssist AI.

Learn the format from the examples.

{few_shot_examples}

Now convert the following clinical note into the SAME format.

Rules:
- Do NOT generate JSON.
- Do NOT explain the answer.
- Do NOT repeat the input.
- Do NOT create another example.
- Only produce:

Symptoms:
- ...

Severity:
...

Input:
{patient_note}

Output:
"""

response = chatBot(
    final_prompt,
    max_new_tokens=50,
    do_sample=False
)

generated_text = response[0]["generated_text"]

# Remove prompt from output
answer = generated_text[len(final_prompt):].strip()

# Remove unwanted text if generated
if "Here is" in answer:
    answer = answer.split("Here is")[0].strip()

if "Output:" in answer:
    answer = answer.split("Output:")[0].strip()

if "Input:" in answer:
    answer = answer.split("Input:")[0].strip()

# Display
print(few_shot_examples)

print("Final Input:\n")
print(patient_note)

print("\nOutput:\n")
print(answer)

Enter Clinical Notes: Patient has cough and fever


[transformers] Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Example 1

Input:
Patient has cough and fever.

Output:
Symptoms:
- Cough
- Fever

Severity:
Moderate

----------------------------------------

Example 2

Input:
Patient complains of headache, nausea, and dizziness.

Output:
Symptoms:
- Headache
- Nausea
- Dizziness

Severity:
Mild

----------------------------------------

Example 3

Input:
Patient has chest pain, shortness of breath, and oxygen level of 88%.

Output:
Symptoms:
- Chest pain
- Shortness of breath

Severity:
Severe

----------------------------------------

Final Input:

Patient has cough and fever

Output:

Symptoms:
- Cough
- Fever

Severity:
Moderate


# Task 4 — Chain of Thought Prompting

A patient has:

- Fever
- Cough
- Oxygen Level = 88%

Design a prompt that encourages the LLM to reason step-by-step before providing the final recommendation.

> **Note:** Prompt design only; do **not** reveal the model's internal reasoning.

In [12]:
COT_prompt = """
You are an experienced medical assistant.

Patient Details:
- Symptoms: Cough, Fever
- Oxygen Level: 88%

Instructions:
- Analyze the patient's condition step by step internally.
- Do NOT reveal your reasoning process.
- Provide only the final medical report.

Response Format:

## Clinical Assessment

## Urgency Level

## Possible Causes

## Actionable Steps

## Immediate Actions (if any)

## Disclaimer
"""

response = chatBot(
    COT_prompt,
    max_new_tokens=200,
    do_sample=False
)

generated_text = response[0]["generated_text"]
answer = generated_text[len(COT_prompt):].strip()

print("Generated Report:\n")
print(answer)

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated Report:

The patient presents with a cough and fever. The oxygen level is 88%, indicating mild hypoxemia. Given these symptoms, the most likely causes include viral or bacterial infections. Further evaluation is needed to determine the exact cause and appropriate treatment.

## Urgency Level
High

## Possible Causes
1. Viral Upper Respiratory Infection
2. Bacterial Pneumonia
3. Bronchitis

## Actionable Steps
1. Order a complete blood count (CBC) and chest X-ray.
2. Administer intravenous fluids if hypotension occurs.
3. Prescribe antipyretics for fever management.
4. Advise the patient to rest and monitor their oxygen levels.

## Immediate Actions (if any)
If the patient develops severe shortness of breath, cyanosis, or persistent high fever, seek immediate medical attention.

## Disclaimer
This assessment is preliminary and should be confirmed with further diagnostic tests. Always consult with a healthcare provider for definitive diagnosis


# Task 5 — Structured Output

Design a prompt that forces the LLM to return **JSON** in the following format:

```json
{
  "disease": "",
  "symptoms": [],
  "risk_level": "",
  "recommendation": ""
}
```

In [13]:
# Task 5: Structured Output

structured_prompt = """
You are MediAssist AI.

Patient Details:
- Fever
- Cough
- Oxygen Level: 88%

Return the response as valid JSON only.

Format:
{
  "disease": "",
  "symptoms": [],
  "risk_level": "",
  "recommendation": ""
}

Do not include markdown, explanations, or any extra text.
"""

response = chatBot(
    structured_prompt,
    max_new_tokens=100,
    do_sample=False
)

generated_text = response[0]["generated_text"]

# Remove the prompt from the output
answer = generated_text[len(structured_prompt):].strip()

print("Generated JSON:\n")
print(answer)

[transformers] Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Generated JSON:

{
  "disease": "Pneumonia",
  "symptoms": ["Fever", "Cough"],
  "risk_level": "High",
  "recommendation": "Seek medical attention immediately"
}


# Task 6 — Model Comparison

Compare any two Hugging Face models.

### Include the following:

- Model Name
- Parameters
- Best Use Case
- Advantages
- Limitations

Present the comparison in a table.

# Solution

| Feature | Llama 3.1 8B | Mistral 7B |
|---------|--------------|------------|
| **Developer** | Meta AI | Mistral AI |
| **Parameters** | 8 Billion | 7 Billion |
| **Best Use Case** | Chatbots, coding assistance, reasoning, content generation | Text generation, summarization, question answering, chatbots |
| **Advantages** | Strong reasoning, high-quality responses, supports long-context tasks, open-source | Fast inference, lightweight, efficient, good performance for its size |
| **Limitations** | Requires higher memory and computational resources | Slightly weaker reasoning than larger models, shorter context window than some newer models |

# Task 7 — Prompt Optimization

The AI produces the following response:

> Maybe the patient has malaria.  
> Or maybe dengue.  
> Or maybe COVID.  
> I am not sure.

### Explain:

- Why is the response poor?
- Which Prompt Engineering techniques would improve it?
- Rewrite the prompt to reduce hallucinations.

# Solution

## 1. Why is the response poor?

- The response is vague and uncertain.
- It suggests multiple diseases without sufficient evidence.
- It provides no reasoning or medical guidance.
- It may lead to misinformation or hallucinations.

## 2. Prompt Engineering techniques to improve it

- **Role Prompting** – Assign the model the role of a professional medical assistant.
- **Context Prompting** – Include the patient's symptoms and relevant medical details.
- **Constraint Prompting** – Instruct the model not to guess or make unsupported diagnoses.
- **Output Formatting** – Require a structured and organized response.

## 3. Optimized Prompt

```text
You are MediAssist AI, a professional healthcare assistant.

Patient Symptoms:
- Fever
- Headache
- Body pain

Instructions:
- Analyze the symptoms carefully.
- Do not guess or make unsupported diagnoses.
- Mention only possible conditions based on the provided symptoms.
- If the information is insufficient, clearly state that more medical information is required.
- Recommend consulting a qualified healthcare professional.

Provide the response in the following format:

1. Possible Conditions
2. Confidence Level
3. Recommendation
4. Disclaimer
```